# 00_b3 — Poland: municipality population by age

Part of the `00_b*` series that replaces the five legacy population tables in the local
`dbis` Postgres (read by `traintran/population.py` / `harmonize_population_lau2.ipynb`) with
newer national-statistics files, each brought to the **same age-band widths** and keyed on
`LAU CODE` from `EU-27-LAU-2025-NUTS-2024.xlsx`.

> **Working notebook — not for publication.** It connects to the local DB only to take the legacy
> table as a baseline; once the harmonised output exists, nothing downstream needs the DB.

| | dataset | location |
|---|---|---|
| legacy (DB) | `public."polandPopulatiaMunicipalties"` | local `dbis` Postgres |
| new (2025) | GUS — *Population. Size and structure and vital statistics in Poland by territorial division, as of 31 December 2025*, **table 12** (gminas by sex and age) (<https://stat.gov.pl/en/topics/population/population/population-size-and-structure-and-vital-statistics-in-poland-by-territorial-division-in-2025-as-of-31-december,3,39.html>) — manual download | `data/eu/ref-national-statistical-offices/pl/ludnosc.stan_i_struktura_31.12.2025.zip` |

**Legacy notes.** Economic bands only: pre-working 0–17, working 18–59 F / 18–64 M, post-working. 180 exact-duplicate rows (2657 → 2477). Join: `municipalityId[:-1]` against `LAU CODE[2:]`.

In [1]:
import os, sys

# Run from the project root so every relative path ('data/...', 'traintran/...') resolves the
# same whether the notebook is launched from notebooks/ or from the root.
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

Working directory: E:\notes\api_zssk\project


In [2]:
import os
from pathlib import Path

import pandas as pd
# import sqlalchemy as sa

import traintran  # noqa: F401  (fixes PROJ on import)
from traintran import population as pop
from traintran.config import default_config

COUNTRY = 'PL'
OUT_DIR = Path('data/eu/ref-national-statistical-offices') / COUNTRY.lower()
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = default_config()
# Database comparison disabled. The 2025 population comes from the statistical office's files below; the
# database held only the older 2021 tables used for a comparison that feeds no output. To re-enable it, set
# TRAINTRAN_PG_URL in the environment and uncomment these lines and the cells marked '[disabled: ...]'.
# engine = sa.create_engine(os.environ['TRAINTRAN_PG_URL'])
print('Country   :', COUNTRY)
print('Output dir:', OUT_DIR)
# print('Postgres  :', cfg.pg_url.rsplit('@', 1)[-1])

Country   : PL
Output dir: data\eu\ref-national-statistical-offices\pl


## 1. Legacy — local DB

The raw table(s) as they sit in `dbis`, geometry columns dropped. A parquet snapshot is written
next to the new source (skip-if-present) so the comparison in §3 can later run without the DB.

In [3]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# GEOM_COLS = {'geom', 'geojson', 'wkb_geometry', 'geometry'}
# LEGACY_TABLES = ['public."polandPopulatiaMunicipalties"']


# def read_table(qualified):
#     schema, table = qualified.split('.', 1)
#     table = table.strip('"')
#     cols = [c['name'] for c in sa.inspect(engine).get_columns(table, schema=schema)
#             if c['name'].lower() not in GEOM_COLS]
#     sel = ', '.join('"' + c.replace('"', '""') + '"' for c in cols)
#     return pd.read_sql(f'select {sel} from {schema}."{table}"', engine)


# legacy_raw = {}
# for t in LEGACY_TABLES:
#     legacy_raw[t] = df = read_table(t)
#     print(f'{t}: {len(df):,} rows x {df.shape[1]} cols')
#     print('  ', list(df.columns))

In [4]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# for t, df in legacy_raw.items():
#     stem = t.replace('.', '_').replace(chr(34), '')
#     snap = OUT_DIR / f'legacy_db_{stem}.parquet'
#     if snap.exists():
#         print('[exists]', snap)
#     else:
#         df.to_parquet(snap, index=False)
#         print('[wrote] ', snap, f'({len(df):,} rows)')

### 1a. Baseline — the current loader on the legacy table

`pop.LOADERS['PL']` is exactly what `harmonize_population_lau2.ipynb` runs today; its keyed and
banded result is what the 2025 data has to reproduce in structure (same `lau_code` coverage,
same harmonised bands).

In [5]:
lau = pop.load_workbook_lau(cfg.lau_file, countries=(COUNTRY,))
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# legacy = pop.harmonise_bands(pop.LOADERS[COUNTRY](engine, lau), COUNTRY)
# legacy['country'] = COUNTRY
# legacy['ref_year'] = 2021

# print(pop.match_report({COUNTRY: legacy}, lau).to_string(index=False))
# print()
# print('band_method  :', legacy.band_method.unique().tolist())
# print('band_complete:', int(legacy.band_complete.sum()), '/', len(legacy))
# print('pop_total    :', f'{legacy.pop_total.sum():,.0f}')
# print()
# print(legacy.head().to_string())

## 2. New (2025) — national statistical office

Source: GUS, *Population. Size and structure and vital statistics in Poland by territorial
division, as of 31 December 2025* (<https://stat.gov.pl/en/topics/population/population/population-size-and-structure-and-vital-statistics-in-poland-by-territorial-division-in-2025-as-of-31-december,3,39.html>). Manual download of the zip into `pl/`
(<https://stat.gov.pl/download/gfx/portalinformacyjny/en/defaultaktualnosci/3286/3/39/1/ludnosc.stan_i_struktura_31.12.2025.zip>).

**Table 12, not table 06.** Table 06 is *Population in powiats by sex and age*: 380 powiats with
4-digit codes. The gmina (LAU) level is **table 12**, *Population in gminas by sex and age*, with
the same layout and 7-digit TERYT codes. (`List of tables.xlsx` in the zip lists all of them.)

Reference date **31 December 2025**, `ref_year = 2025`, the same date as CZ, SK and HU. (The earlier
30 June 2026 release, `population._size_and_structure_vi_2026.zip`, is no longer read.)

In [6]:
NEW_FILE = OUT_DIR / 'ludnosc.stan_i_struktura_31.12.2025.zip'

if NEW_FILE is not None and NEW_FILE.exists():
    print('[exists]', NEW_FILE, f'({NEW_FILE.stat().st_size / 1e6:.1f} MB)')
else:
    print('[missing] no direct link — download manually from the source page in the title cell')
    print('          and save it into', OUT_DIR)

[exists] data\eu\ref-national-statistical-offices\pl\ludnosc.stan_i_struktura_31.12.2025.zip (10.4 MB)


In [7]:
import zipfile

PL_TABLE = 'tabela12.xls'
PL_XLS = OUT_DIR / f'{NEW_FILE.stem}_{PL_TABLE}'   # tied to its zip: no vintage mix-up
if PL_XLS.exists():
    print('[exists]', PL_XLS)
else:
    with zipfile.ZipFile(NEW_FILE) as z:
        PL_XLS.write_bytes(z.read(PL_TABLE))
    print('[unzipped]', PL_XLS, f'({PL_XLS.stat().st_size / 1e6:.1f} MB)')

[exists] data\eu\ref-national-statistical-offices\pl\ludnosc.stan_i_struktura_31.12.2025_tabela12.xls


### 2a. Raw sheets

One sheet per voivodship, printed one gmina block after another. Column A holds **both** the
gmina name and, below it, the age labels; column B (TERYT code) is filled **only on the name
row**; column C is the total of both sexes (`-` means zero). Each block runs: name row (the gmina
total) → 5-year groups `0 - 4` … `80 - 84`, `85 i więcej`, with single years under the first six →
economic groups (*Wiek przedprodukcyjny / produkcyjny / poprodukcyjny*) → *Biologiczne grupy wieku*
(0–14 / 15–64 / 65+) → education groups.

Only columns A–C are read.

In [8]:
xls = pd.ExcelFile(PL_XLS, engine='xlrd')
parts = []
for sheet in xls.sheet_names:
    s = pd.read_excel(xls, sheet_name=sheet, header=None, dtype=str, usecols=[0, 1, 2])
    s.columns = ['label', 'code', 'value']
    s['voivodship'] = sheet
    s['row'] = range(len(s))
    parts.append(s)
pl_raw = pd.concat(parts, ignore_index=True)
print(f'{len(xls.sheet_names)} sheets, {len(pl_raw):,} rows')
pl_raw.iloc[8:40]

16 sheets, 178,480 rows


,label,code,value,voivodship,row
8,Jelenia Góra,0261011,73470,Dolnośląskie,8
9,0 - 4,,2073,Dolnośląskie,9
10,0,,326,Dolnośląskie,10
11,1,,327,Dolnośląskie,11
12,2,,434,Dolnośląskie,12
13,3,,456,Dolnośląskie,13
14,4,,530,Dolnośląskie,14
15,5 - 9,,2856,Dolnośląskie,15
16,5,,488,Dolnośląskie,16
17,6,,590,Dolnośląskie,17


### 2b. Harmonise — one row per gmina

1. Every row inherits the TERYT code of the name row above it (forward-fill within the sheet).
2. The **5-year groups** are summed from column C into the three target bands: lower bound < 15 →
   `popAge0to14`, < 65 → `popAge15to64`, the rest (`65 - 69` … `85 i więcej`) → `popAge65over`.
3. Checks, all asserted: 18 five-year groups per block; their sum equals the block total; the
   three summed bands equal GUS's own *Biologiczne grupy wieku* rows; the economic groups also add
   up to the total.
4. The economic groups are kept as `preworkingAge` / `workingAge` / `postworkingAge`, the legacy
   table's names, for a like-for-like comparison with it.

The TERYT type digit (7th) separates gminas (1 urban, 2 rural, 3 urban-rural) from **city
districts** (8 Warsaw dzielnice, 9 delegatury of Łódź, Poznań, Wrocław). The districts
are not LAUs (their city is already a gmina row), so they are kept in `pl_districts` and dropped.

In [9]:
import re

r = pl_raw.copy()
r['label'] = r['label'].fillna('').str.replace(r'\s+', ' ', regex=True).str.strip()
r['code'] = r['code'].fillna('').str.strip()
is_head = r['code'].str.fullmatch(r'\d{7}')
r['teryt'] = r['code'].where(is_head).groupby(r['voivodship']).ffill()
r = r[r['teryt'].notna()].copy()
r['n'] = pd.to_numeric(r['value'].str.strip().replace('-', '0'), errors='coerce')

head = r[is_head.loc[r.index]].rename(columns={'label': 'municipalityName', 'n': 'popTotal'})
assert head['teryt'].is_unique, 'TERYT repeated'

# 5-year groups 'a - b' plus one open-ended top group 'N i więcej'. N is 85 for most gminas,
# but small ones are top-coded at 80 or 75: fine here, as every open group starts at >= 65.
m = r['label'].str.extract(r'^(\d+) ?- ?(\d+)$')
top = r['label'].str.extract(r'^(\d+) i więcej$')[0]
five = r[m[0].notna() | top.notna()].copy()
five['lo'] = pd.to_numeric(m.loc[five.index, 0].fillna(top.loc[five.index])).astype(int)
five['hi'] = pd.to_numeric(m.loc[five.index, 1]).fillna(999).astype(int)
five['band'] = pd.cut(five['lo'], [-1, 14, 64, 200], labels=['popAge0to14', 'popAge15to64', 'popAge65over'])

# contiguous from 0, exactly one open group, and that group starts at >= 65
g = five.sort_values(['teryt', 'lo']).groupby('teryt')
ok = g.apply(lambda x: x['lo'].iloc[0] == 0 and (x['lo'].iloc[1:].values == x['hi'].iloc[:-1].values + 1).all()
             and (x['hi'] == 999).sum() == 1 and x['hi'].iloc[-1] == 999 and x['lo'].iloc[-1] >= 65)
assert ok.all(), ok[~ok]
open_top = g['lo'].max().value_counts().sort_index()
bands = five.pivot_table(index='teryt', columns='band', values='n', aggfunc='sum', observed=True)

LABELS = {'Wiek przedprodukcyjny': 'preworkingAge', 'Wiek produkcyjny': 'workingAge',
          'Wiek poprodukcyjny': 'postworkingAge',
          '0-14 lat': 'bio0to14', '15-64 lat': 'bio15to64', '65 lat i więcej': 'bio65over'}
named = (r[r['label'].isin(LABELS)].assign(col=lambda x: x['label'].map(LABELS))
         .pivot(index='teryt', columns='col', values='n'))

pl_all = (head[['teryt', 'municipalityName', 'voivodship', 'popTotal']]
          .merge(bands, left_on='teryt', right_index=True)
          .merge(named, left_on='teryt', right_index=True))
pl_all['teryt_type'] = pl_all['teryt'].str[-1]

B = ['popAge0to14', 'popAge15to64', 'popAge65over']
E = ['preworkingAge', 'workingAge', 'postworkingAge']
assert (pl_all[B].sum(axis=1) == pl_all.popTotal).all(), '5-year groups do not add up to the total'
assert (pl_all[B].values == pl_all[['bio0to14', 'bio15to64', 'bio65over']].values).all(), 'differs from GUS bands'
assert (pl_all[E].sum(axis=1) == pl_all.popTotal).all(), 'economic groups do not add up'

pl_districts = pl_all[pl_all.teryt_type.isin(['8', '9'])]
pl_new = (pl_all[~pl_all.teryt_type.isin(['8', '9'])]
          .drop(columns=['bio0to14', 'bio15to64', 'bio65over']).reset_index(drop=True))
pl_new = pl_new.astype({c: 'int64' for c in ['popTotal', *B, *E]})
pl_new['ref_year'] = 2025
pl_new['ref_date'] = '2025-12-31'

print(f'blocks parsed: {len(pl_all):,}  ->  gminas {len(pl_new):,} + city districts {len(pl_districts)}')
print('gmina types   :', pl_new.teryt_type.value_counts().sort_index().to_dict(), '(1 urban, 2 rural, 3 urban-rural)')
print(f'population    : {pl_new.popTotal.sum():,}')
print('top-coded at  :', {f'{k}+': int(v) for k, v in open_top.items()}, '(blocks)')
print('all checks passed: 5-year sum = total, bands = GUS bands, economic groups = total')
pl_new.head()

blocks parsed: 2,512  ->  gminas 2,479 + city districts 33
gmina types   : {'1': 302, '2': 1459, '3': 718} (1 urban, 2 rural, 3 urban-rural)
population    : 37,332,510
top-coded at  : {'85+': 2512} (blocks)
all checks passed: 5-year sum = total, bands = GUS bands, economic groups = total


,teryt,municipalityName,voivodship,popTotal,popAge0to14,popAge15to64,popAge65over,postworkingAge,preworkingAge,workingAge,teryt_type,ref_year,ref_date
0,0261011,Jelenia Góra,Dolnośląskie,73470,7956,44574,20940,23429,9996,40045,1,2025,2025-12-31
1,0262011,Legnica,Dolnośląskie,89919,11367,56277,22275,25240,14135,50544,1,2025,2025-12-31
2,0264011,Wrocław,Dolnośląskie,672601,89501,442608,140492,156515,106993,409093,1,2025,2025-12-31
3,0265011,Wałbrzych,Dolnośląskie,97546,10392,59944,27210,30731,13184,53631,1,2025,2025-12-31
4,0224013,Bardo,Dolnośląskie,4781,538,3126,1117,1298,686,2797,3,2025,2025-12-31


In [10]:
# The city districts must add up to their city, otherwise something was mis-assigned.
cities = {'1465': 'Warszawa', '1061': 'Łódź', '3064': 'Poznań', '0264': 'Wrocław'}
for p4, name in cities.items():
    dist = pl_districts[pl_districts.teryt.str[:4] == p4]
    city = pl_new[pl_new.teryt.str[:4] == p4]
    if len(dist):
        print(f'{name:<9} {len(dist):>2} districts = {dist.popTotal.sum():>9,.0f}   city row {city.popTotal.sum():>9,}')

Warszawa  18 districts = 1,866,729   city row 1,866,729
Łódź       5 districts =   639,890   city row   639,890
Poznań     5 districts =   534,239   city row   534,239
Wrocław    5 districts =   672,601   city row   672,601


### 2c. Joinability — legacy BDL id vs new TERYT, and both vs the 2025 workbook

The workbook's PL `LAU CODE` is **not** TERYT: it is `'10'` + the legacy BDL unit id without its
last digit (that is what `pop.pl_key` / `pl_lau_key` already rely on). Its TERYT parts sit at fixed
positions — voivodship `[4:6]`, powiat `[9:11]`, gmina `[11:13]` — while the TERYT type digit is
dropped. So the new table joins on the **first six TERYT digits**.

Example: Bochnia (urban) TERYT `1201011` ↔ LAU `10 01 12 12 0 01 01` → `12`·`01`·`01`.

In [11]:
def lau_to_teryt6(lau_code):
    return lau_code[4:6] + lau_code[9:11] + lau_code[11:13]


lau['teryt6'] = lau['lau_code'].map(lau_to_teryt6)
assert lau['teryt6'].is_unique
pl_new['teryt6'] = pl_new['teryt'].str[:6]
assert pl_new['teryt6'].is_unique

pl_new_lau = pl_new.merge(lau[['teryt6', 'lau_code', 'lau_name_national']], on='teryt6', how='left')
print(pop.match_report({COUNTRY: pl_new_lau}, lau).to_string(index=False))

print('\nnew gminas not in the 2025 workbook:')
print(pl_new_lau.loc[pl_new_lau.lau_code.isna(), ['teryt', 'municipalityName', 'voivodship', 'popTotal']]
      .to_string(index=False) or '  none')
wb = set(lau['lau_code'])
print('\nworkbook LAU codes not covered by the new file:')
print(lau.loc[~lau.lau_code.isin(pl_new_lau.lau_code), ['lau_code', 'lau_name_national']]
      .to_string(index=False) or '  none')

renamed = pl_new_lau[pl_new_lau.lau_code.notna() &
                     (pl_new_lau.municipalityName != pl_new_lau.lau_name_national)]
print(f'\nsame code, different name: {len(renamed)}')
print(renamed[['teryt', 'municipalityName', 'lau_name_national']].head(15).to_string(index=False))

country  source_rows  workbook_lau  matched  source_unmatched  lau_uncovered
     PL         2479          2477     2477                 0              0

new gminas not in the 2025 workbook:
  teryt municipalityName  voivodship  popTotal
1207132          Szczawa Małopolskie      1799
2002162         Grabówka   Podlaskie     10852

workbook LAU codes not covered by the new file:
Empty DataFrame
Columns: [lau_code, lau_name_national]
Index: []

same code, different name: 0
Empty DataFrame
Columns: [teryt, municipalityName, lau_name_national]
Index: []


In [12]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# # Legacy BDL id -> TERYT7 directly (woj [2:4], powiat [7:9], gmina+type [9:12]).
# old = legacy_raw['public."polandPopulatiaMunicipalties"'].drop_duplicates('municipalityId').copy()
# old['teryt'] = old.municipalityId.str[2:4] + old.municipalityId.str[7:9] + old.municipalityId.str[9:12]
# old['teryt6'] = old['teryt'].str[:6]
# j = old[['teryt', 'teryt6', 'municipalityName']].merge(
#     pl_new[['teryt', 'teryt6', 'municipalityName']], on='teryt6', how='outer',
#     suffixes=('_old', '_new'), indicator=True)
# print(j['_merge'].value_counts().to_string())
# both = j[j._merge == 'both']
# changed = both[both.teryt_old != both.teryt_new]
# print(f'\nsame gmina, TERYT type digit changed (e.g. village granted town rights): {len(changed)}')
# print(changed.head(10).to_string(index=False))
# print('\nonly in new:')
# print(j.loc[j._merge == 'right_only', ['teryt_new', 'municipalityName_new']].to_string(index=False))

### 2d. Territorial changes — 2026 splits folded back into their 2025 parent

On **1 Jan 2026** two gminas were carved out of existing ones. The 2025 workbook and the LAU 2024
geometry only know the parent, so a new gmina, if present, is **added back into its parent's LAU**
(same rule as Austria: keep the larger unit both sides can describe); `includes_teryt` lists the
TERYT codes added to a row, and the assert requires each new gmina to sit in its parent's powiat.

| new gmina (TERYT) | split from (workbook LAU) |
|---|---|
| Szczawa `1207132` | Kamienica `1001121220705` |
| Grabówka `2002162` | Supraśl `1006201370209` |

GUS publishes the **31 Dec 2025** figures in the territorial division of **1 Jan 2026**, so both new
gminas are in this release and are folded. The fold applies only to codes actually present.

In [13]:
# new TERYT (2026) -> parent LAU code (2025 workbook)
PL_SPLITS_KNOWN = {'1207132': '1001121220705', '2002162': '1006201370209'}
PL_SPLITS = {t: p for t, p in PL_SPLITS_KNOWN.items() if t in set(pl_new_lau.teryt)}
print(f'splits present in this release: {len(PL_SPLITS)} of {len(PL_SPLITS_KNOWN)} known')

lau_t6 = lau.set_index('lau_code')['teryt6']
for t, parent in PL_SPLITS.items():
    assert parent in lau_t6.index, parent
    assert t[:4] == lau_t6[parent][:4], f'{t} is not in the powiat of {parent}'

num = ['popTotal', *B, *E]
x = pl_new_lau.copy()
x['lau_code'] = x['lau_code'].fillna(x['teryt'].map(PL_SPLITS))
assert x['lau_code'].notna().all(), x.loc[x.lau_code.isna(), ['teryt', 'municipalityName']]

before = x[x.lau_code.isin(PL_SPLITS.values())][['lau_code', 'teryt', 'municipalityName', 'popTotal']]
sums = x.groupby('lau_code')[num].sum()
added = x[x.teryt.isin(PL_SPLITS)].groupby('lau_code')['teryt'].agg(';'.join)
base = x[~x.teryt.isin(PL_SPLITS)].set_index('lau_code')
base[num] = sums.loc[base.index]
base['includes_teryt'] = added.reindex(base.index)
pl_new_lau = base.reset_index()

assert pl_new_lau['lau_code'].is_unique and set(pl_new_lau.lau_code) == set(lau.lau_code)
assert pl_new_lau['popTotal'].sum() == pl_new['popTotal'].sum(), 'population lost in the fold'
print(f'pl_new_lau: {len(pl_new_lau):,} rows = workbook LAUs {lau.lau_code.nunique():,}; '
      f'population unchanged {pl_new_lau.popTotal.sum():,}')
print('\nbefore folding:')
print(before.sort_values(['lau_code', 'teryt']).to_string(index=False))
print('\nafter folding:')
print(pl_new_lau.loc[pl_new_lau.includes_teryt.notna(),
                     ['lau_code', 'teryt', 'municipalityName', 'popTotal', 'includes_teryt']].to_string(index=False))

splits present in this release: 2 of 2 known
pl_new_lau: 2,477 rows = workbook LAUs 2,477; population unchanged 37,332,510

before folding:
     lau_code   teryt municipalityName  popTotal
1001121220705 1207052        Kamienica      5948
1001121220705 1207132          Szczawa      1799
1006201370209 2002093          Supraśl      8188
1006201370209 2002162         Grabówka     10852

after folding:
     lau_code   teryt municipalityName  popTotal includes_teryt
1001121220705 1207052        Kamienica      7747        1207132
1006201370209 2002093          Supraśl     19040        2002162


In [14]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# # Same LAU, two vintages. Legacy has only the economic groups, so compare those like for like.
# cmp = (legacy[['lau_code', 'pop_total', 'pl_preworking', 'pl_working', 'pl_postworking']]
#        .merge(pl_new_lau.dropna(subset=['lau_code'])
#               [['lau_code', 'municipalityName', 'popTotal', *E, *B]],
#               on='lau_code', how='outer', indicator=True))
# print(cmp['_merge'].value_counts().to_string())
# b = cmp[cmp._merge == 'both'].copy()
# b['diff_%'] = 100 * (b.popTotal - b.pop_total) / b.pop_total
# print(f'\ntotal  legacy {b.pop_total.sum():,.0f}  ->  new {b.popTotal.sum():,.0f}')
# print(b['diff_%'].describe().round(2).to_string())
# print('\neconomic-group shares (%), legacy -> new:')
# for o, n in zip(['pl_preworking', 'pl_working', 'pl_postworking'], E):
#     print(f'  {n:<15} {100 * b[o].sum() / b.pop_total.sum():5.1f} -> {100 * b[n].sum() / b.popTotal.sum():5.1f}')
# print('\nwhat the exact bands change vs the legacy approx_econ mapping (new data, %):')
# for e, x in zip(E, B):
#     print(f'  {e:<15} {100 * b[e].sum() / b.popTotal.sum():5.1f}   vs {x:<13} {100 * b[x].sum() / b.popTotal.sum():5.1f}')
# print('\nlargest relative changes:')
# print(b.reindex(b['diff_%'].abs().sort_values(ascending=False).index)
#        .head(10)[['lau_code', 'municipalityName', 'pop_total', 'popTotal', 'diff_%']].round(1).to_string(index=False))

### 2e. Join to the v4atua LAU subset (GeoJSON)

Polish features only (`CNTR_CODE = 'PL'`); `GISCO_ID = 'PL_<LAU code>'`, the same 13-digit code as
the workbook.

In [15]:
import geopandas as gpd

LAU_SUBSET = Path('data/eu/ref-lau-2024-01m/subsetv4atua/LAU_RG_01M_2024_4326.geojson')
pl_geo = gpd.read_file(LAU_SUBSET, where="CNTR_CODE = 'PL'")
pl_geo['lau_code'] = pl_geo['GISCO_ID'].str.split('_', n=1).str[1]
assert pl_geo['lau_code'].is_unique
print(f'{LAU_SUBSET.name}: {len(pl_geo):,} PL features')

jg = pl_geo.drop(columns='geometry').merge(
    pl_new_lau[['lau_code', 'teryt', 'municipalityName', 'popTotal']],
    on='lau_code', how='outer', indicator=True)
print('\n' + jg['_merge'].value_counts().to_string())
print('\ngeojson only:', jg.loc[jg._merge == 'left_only', 'lau_code'].tolist())
print('new file only:', jg.loc[jg._merge == 'right_only', ['teryt', 'municipalityName']].values.tolist())

m = jg[jg._merge == 'both'].copy()
m['diff_%'] = 100 * (m.popTotal - m.POP_2024) / m.POP_2024
print(f'\npopulation  GISCO POP_2024 {m.POP_2024.sum():,.0f}  vs  new popTotal {m.popTotal.sum():,.0f}')
print(m['diff_%'].describe().round(2).to_string())
print('\nlargest relative changes:')
print(m.reindex(m['diff_%'].abs().sort_values(ascending=False).index)
       .head(8)[['lau_code', 'LAU_NAME', 'POP_2024', 'popTotal', 'diff_%']].round(1).to_string(index=False))

LAU_RG_01M_2024_4326.geojson: 2,477 PL features

_merge
both          2477
left_only        0
right_only       0

geojson only: []
new file only: []

population  GISCO POP_2024 37,636,508  vs  new popTotal 37,332,510
count    2477.00
mean       -1.10
std         1.62
min        -6.04
25%        -2.10
50%        -1.38
75%        -0.41
max         7.74

largest relative changes:
     lau_code         LAU_NAME  POP_2024  popTotal  diff_%
1007141301803       Lesznowola   38479.0     41459     7.7
1003021042301         Czernica   23974.0     25808     7.6
1003021041803         Miękinia   21613.0     23144     7.1
1004221401105         Kosakowo   21436.0     22929     7.0
1007141300506       Żabia Wola   11096.0     11863     6.9
1002301612106       Kleszczewo   11641.0     12420     6.7
1002301591302            Lipno    9918.0     10545     6.3
1003021010604 Szklarska Poręba    5776.0      5427    -6.0


## 3. Write — common-schema dump

One CSV per country, same columns in all five `00_b*` notebooks, read by
`00_extract_base_data_eu.ipynb` §2c to attach the 2025 population to the LAU subset GeoJSON.

| column | content |
|---|---|
| `lau_code`, `gisco_id` | LAU code (text) and `<CC>_<lau_code>`, the subset key |
| `country`, `lau_name` | country code, national name |
| `pop_total`, `y_0_14`, `y_15_64`, `y_65plus` | persons (`pop.HARMONISED_BANDS`) |
| `band_method`, `ref_year`, `ref_date`, `source` | `exact`; reference year/date; dataset id |
| `native_id` | the source's own key |
| `replaces_lau` | workbook LAU codes a merged row covers (`;`-joined), else empty |
| `includes_native` | native units summed into the row (`;`-joined), else empty |

Skips if the file exists; set `REWRITE_DUMP = True` to overwrite. Read it back with `dtype=str`
for the code columns (HU codes have leading zeros).

In [16]:
REWRITE_DUMP = False
DUMP_COLS = ['lau_code', 'gisco_id', 'country', 'lau_name', 'pop_total', *pop.HARMONISED_BANDS,
             'band_method', 'ref_year', 'ref_date', 'source', 'native_id', 'replaces_lau', 'includes_native']
DUMP_FILE = OUT_DIR / f'lau_population_2025_{COUNTRY.lower()}.csv'

f = pl_new_lau.reset_index(drop=True)
dump = pd.DataFrame({
    'lau_code': f['lau_code'].astype(str),
    'gisco_id': COUNTRY + '_' + f['lau_code'].astype(str),
    'country': COUNTRY,
    'lau_name': f['municipalityName'],
    'pop_total': f['popTotal'], 'y_0_14': f['popAge0to14'], 'y_15_64': f['popAge15to64'], 'y_65plus': f['popAge65over'],
    'band_method': 'exact',
    'ref_year': f['ref_year'], 'ref_date': f['ref_date'],
    'source': 'GUS_tab12_2025-12-31',
    'native_id': f['teryt'].astype(str),
    'replaces_lau': None,
    'includes_native': f['includes_teryt'],
})[DUMP_COLS]

assert dump['lau_code'].is_unique and dump['gisco_id'].is_unique
assert (dump[pop.HARMONISED_BANDS].sum(axis=1) == dump['pop_total']).all(), 'bands do not add up'
assert dump[['pop_total', *pop.HARMONISED_BANDS]].notna().all().all()
replaced = {o for r in dump['replaces_lau'].dropna() for o in r.split(';')}
merged_new = set(dump.loc[dump['replaces_lau'].notna(), 'lau_code'])
assert (set(dump['lau_code']) - merged_new) | replaced == set(lau['lau_code']), 'dump does not cover the workbook LAUs'

if DUMP_FILE.exists() and not REWRITE_DUMP:
    print('[exists]', DUMP_FILE, '-- set REWRITE_DUMP = True to overwrite')
else:
    dump.to_csv(DUMP_FILE, index=False, encoding='utf-8')
    print('[wrote] ', DUMP_FILE)
back = pd.read_csv(DUMP_FILE, dtype={'lau_code': str, 'gisco_id': str, 'native_id': str})
assert list(back.columns) == DUMP_COLS and back['lau_code'].tolist() == dump['lau_code'].tolist(), 'file differs from dump'
print(f'{len(back):,} rows, population {back.pop_total.sum():,}; '
      f"merged rows {int(back.replaces_lau.notna().sum())}, aggregated rows {int(back.includes_native.notna().sum())}")
back.head(3)

[exists] data\eu\ref-national-statistical-offices\pl\lau_population_2025_pl.csv -- set REWRITE_DUMP = True to overwrite
2,477 rows, population 37,332,510; merged rows 0, aggregated rows 2


,lau_code,gisco_id,country,lau_name,pop_total,y_0_14,y_15_64,y_65plus,band_method,ref_year,ref_date,source,native_id,replaces_lau,includes_native
0,1003021016101,PL_1003021016101,PL,Jelenia Góra,73470,7956,44574,20940,exact,2025,2025-12-31,GUS_tab12_2025-12-31,0261011,NaN,NaN
1,1003021026201,PL_1003021026201,PL,Legnica,89919,11367,56277,22275,exact,2025,2025-12-31,GUS_tab12_2025-12-31,0262011,NaN,NaN
2,1003021056401,PL_1003021056401,PL,Wrocław,672601,89501,442608,140492,exact,2025,2025-12-31,GUS_tab12_2025-12-31,0264011,NaN,NaN
